# 02 · Cruce emocional × espacial

Une las referencias espaciales (notebook 01) con la clasificación afectiva del corpus para producir:

- Tabla pivote lugar × categoría con índice de desencanto
- Distribución por zona urbana y categoría socio-espacial
- Mapa de calor afectivo (PNG)
- GeoJSON con los lugares y sus emociones
- HTML interactivo con filtros

In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / 'src'))

import corpus, spatial_extraction, affect_loader, cross_analysis
import exporters, viz_maps, interactive

CONFIG = 'configs/sin_remedio.yaml'
CORPUS = 'data/corpus.txt'
AFECTIVO = 'data/extraccion_emocional.xlsx'
cfg = corpus.load_config(CONFIG)

## 1. Reconstruir la extracción espacial (o cargar la del notebook 01)

In [ ]:
parrafos = corpus.parse_from_config(CORPUS, cfg)
with open(cfg['lugares_catalogo_path'], encoding='utf-8') as f:
    catalogo = json.load(f)
df_esp = spatial_extraction.extract(parrafos, catalogo)
print(f"Referencias espaciales: {len(df_esp)}")

## 2. Cargar la extracción afectiva

In [ ]:
df_emo = affect_loader.load(AFECTIVO)
print(f"Párrafos clasificados: {len(df_emo)}")
df_emo.head(3)

## 3. Cruzar

In [ ]:
cruce = cross_analysis.cross_spatial_affect(df_esp, df_emo)
cobertura = cruce['tiene_clasif_afectiva'].mean() * 100
print(f"Cobertura: {cobertura:.1f}% de referencias con clasificación afectiva")

## 4. Formato largo y tablas pivote

In [ ]:
polaridad_fn = cross_analysis.make_polarity_fn(cfg['polaridad'])
df_largo = cross_analysis.long_format(cruce, polaridad_fn)

pivot_lugar = cross_analysis.pivot_place_category(
    df_largo,
    cfg['polaridad']['Negativa'],
    cfg['polaridad']['Positiva'],
    cfg['polaridad']['Tensión social'],
)
pivot_lugar.head(15)

In [ ]:
pivot_social = cross_analysis.pivot_social_category(
    df_largo, cfg['categorias_sociales'])
pivot_social

## 5. Mapa de calor afectivo

In [ ]:
# Preparar lista de lugares con sus métricas para el mapa
places = []
coords = df_esp.drop_duplicates('Lugar nombrado (canónico)').set_index(
    'Lugar nombrado (canónico)')[['Latitud (aprox.)', 'Longitud (aprox.)', 'Tipo de lugar', 'Zona']].to_dict('index')

for _, row in pivot_lugar.iterrows():
    info = coords.get(row['Lugar'], {})
    places.append({
        'name': row['Lugar'], 'lugar': row['Lugar'],
        'tipo': row['Tipo'], 'zona': row['Zona'],
        'lat': info.get('Latitud (aprox.)'),
        'lon': info.get('Longitud (aprox.)'),
        'menciones_total': int(row['Total_menciones_cat']),
        'total_menciones': int(row['Total_menciones_cat']),
        'negativas': int(row['Negativas']),
        'positivas': int(row['Positivas']),
        'tension_social': int(row['Tension_social']),
        'idx_desencanto': row['Indice_desencanto'],
        'indice_desencanto': row['Indice_desencanto'],
    })

viz_maps.heat_map(places, 'outputs/mapa_calor_afectivo.png',
                  geom=cfg['geometria_base'],
                  title='Mapa de calor afectivo · ' + cfg['proyecto']['nombre'])

## 6. GeoJSON y HTML interactivo

In [ ]:
exporters.to_geojson(places, 'outputs/lugares_emocional.geojson')

# Para el HTML, anexar también las categorías por lugar
for p in places:
    row = pivot_lugar[pivot_lugar['Lugar'] == p['lugar']].iloc[0] if (pivot_lugar['Lugar']==p['lugar']).any() else None
    cats = {}
    if row is not None:
        for c in pivot_lugar.columns:
            if c not in ('Lugar','Tipo','Zona','Total_menciones_cat','Negativas','Positivas','Tension_social','Indice_desencanto'):
                v = row[c]
                if isinstance(v,(int,float)) and v>0: cats[c]=int(v)
    p['categorias'] = cats
    p['supuesto_real'] = ''  # opcional

interactive.affective_map_html(places, 'outputs/mapa_interactivo.html')
print('✓ HTML generado')

## 7. Exportar Excel completo

In [ ]:
exporters.to_excel({
    'Cruce párrafo a párrafo': {'df': cruce.drop(columns=[c for c in cruce.columns if c.endswith('_list')]), 'title': 'Cruce párrafo a párrafo'},
    'Lugar × Categoría': {'df': pivot_lugar, 'title': 'Pivote lugar × categoría', 'conditional': ['Indice_desencanto']},
    'Cat social × Polaridad': {'df': pivot_social, 'title': 'Categoría socio-espacial'},
}, 'outputs/02_cruce_espacial_emocional.xlsx')